# Burnback simulation

essentially continuation of [Burning](Burning.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from duckdb import sql as sqldf


from Capstone.Rockets.Plots import dots_and_arrows, Interactive_polar, Shape
from Capstone.Rockets.Simulation import run, step, grain
from Capstone.Rockets.Simulation import HScurves, HSintersections, HSsim
from Capstone.Superformula.Formulas import formula1, formula2
from Capstone.Geometry import cart2pol, pol2cart, rad2deg, normals, magn
from Capstone.utils import describe, clip, shape, pick, parse_sf_params

## Simulation run

[v] Clear Caustics  
[ ] Intersection point as replacement for caustic  
[v] Intersection window processes the whole closed circle  
[v] Fill Rarefactions (primitive interpolation)  
[ ] Rarefactions - better interpolation  
[ ] Interpolate multiple point in between rarefactions  
[v] Points stop advancing when front reaches casing  
[ ] Points stop exactly on the casing  
[v] Separation of burning regions  
[ ] Handling cusps - regions that separate entirely from front [?]  
[v] Harvester - gathers data during sim for dbg and visualization  
[v] Interactive plot - animation of burning front  
[ ] Dots and arrows plot - zoom into data of specific steps for debugging  
[ ] Compare results and timings with Kang simulation.  

In [ ]:


#fn = "superformula_funcname=superformula_m=10_a=1_b=0.5_n_1=74.1_n_2=1.51_n_3=35.5_s=0_o=0_invert=0_20260521_180513.png"
fn = "superformula_funcname=trapezoid_m=4.4_a=1_b=1.5_n_1=22.4_n_2=4.17_n_3=10.2_s=0_o=3.8_invert=1_20260602_181155.png"
#fn = "superformula_funcname=risingsun_m=5_a=1_b=1_n_1=5.01_n_2=100_n_3=100_20260427_123859.png"

cap = parse_sf_params(fn)
cap

In [ ]:

d = .1
steps = 40
n = 1000

trw = {
"funcname":"trapez wave",
"m":5, "a":1,
"n_1":-0.66, "n_2":0.95,
"s":0, "o":1, "invert":1
}

sfl = {
"funcname":"superformula",
"m":5, "a":1,
"n_1":-0.61, "n_2":0.94,
"s":2, "o":1, "invert":1
}


profile = formula1(**trw)


In [ ]:
I, R, T, X, Y = grain(profile, 1000)
Shape(R,T)

In [ ]:

run(profile, d, steps, n, 4)
print("Simulation completed.")


In [ ]:

XY = np.array([[ 2.19982715e+00, -2.75607139e-02],
       # [ 2.19995775e+00, -1.35425189e-02],
       [ 2.19999894e+00, -8.47615822e-04],
       [ 2.09592584e+00,  1.74919036e-01],
       [ 2.06291790e+00,  1.96024262e-01],
       [ 2.04085989e+00,  2.09070462e-01]])


xy =  np.array([[2.14796239, 0.08703571]])


xyi =  np.array([[  2.14796239, 268.40670123]])

In [ ]:
xy.shape

In [ ]:
from Capstone.Rockets.Simulation import interpolate

In [ ]:
ln = np.linspace(2.04, 2.2, 20)
ln.shape

In [ ]:
ln = np.stack([ln,ln],axis=1)
ln.shape

In [ ]:
res = interpolate(XY, ln)
res

In [ ]:
a = np.concatenate([XY, xy, xyi, res ], axis=0)

In [ ]:
apd = pd.DataFrame(a, columns=['X','Y'])
apd['cls'] = ['XY']*len(XY) + ['xy']*len(xy) + ['xyi']*len(xyi) + ['res']*len(res)
apd

In [ ]:
fig = px.scatter(apd, x='X', y='Y', color = 'cls')

fig

In [ ]:

HSdata = HScurves.results()
HSintrsctns = HSintersections.results()
HSsimdata = HSsim.results()

describe(HSdata)
describe(HSintrsctns)

describe(HSsimdata)


In [ ]:
StepsFilter = (0,1,2,3)

HSdataSmol = HScurves.results(S=StepsFilter)
HSintrsctnsSmol = HSintersections.results(SimStep=StepsFilter)

describe(HSdataSmol)
describe(HSintrsctnsSmol)

In [ ]:
tmp = HSdataSmol.copy()

tmp.update(HSintrsctnsSmol)
tmp.update(HSsimdata)

describe(tmp)


In [ ]:
dots_and_arrows(**tmp)

In [ ]:
hsdf = pd.DataFrame(HSdata)
hsdf

In [ ]:
hsdf.S.unique()

In [ ]:
q = """ --
select *
from hsdf
where S in (2,3) and (I < 10 or I > 700)
-- group by item , tbl
--order by non_null_values desc
"""

wat = sqldf(q).df()

wat

In [ ]:
px.scatter(wat, x='X', y='Y', color='isNew',  hover_data=['S','I','X','Y'], width=800, height=600)

In [ ]:


def burnback(I, X, Y, Nx, Ny, I1, X1, Y1, S, Ex, Ey, Px, Py, clas, filt, **kwargs):
    fig = go.Figure()
   
    fig.add_trace(go.Scatter(x=X1, y=Y1, mode='lines', marker=dict(size=6, color=filt*1),
                             hovertext=S, name='Front'))
        
    Hx = kwargs['Hx']
    Hy = kwargs['Hy']
    fig.add_trace(go.Scatter(x=Hx, y=Hy, mode='lines', marker=dict(size=6, color=filt*1),
                             name='Hull'))

    fig.update_layout(width=800, height=800)
    # fig.update_xaxes(range=roi['x'])
    # fig.update_yaxes(range=roi['y'])
    fig.show()

In [ ]:
burnback(**tmp)

In [ ]:
full = HSdata.copy()

full.update(HSintrsctns)
full.update(HSsimdata)

describe(full)


In [ ]:
", ".join(tmp.keys())

In [ ]:
I, X, Y, A, S, isNew, Nx, Ny, Ex, Ey, filt, SimStep, ii, ti, tj, Px, Py, clas, condi, condj, s, C, Hx, Hy = full.values()

In [ ]:
toplot = {k:full[k] for  k in ['I', 'X', 'Y', 'A', 'S', 'isNew']}

In [ ]:
describe(toplot)

In [ ]:
df = pd.DataFrame(toplot)
df.isNew = df.isNew.astype(bool)
df

In [ ]:
df.isNew[0] = True # so the animation has both colors in the first frame

In [ ]:
#R upper bound
Rub = np.ceil(Hx.max()) 

fig = px.scatter(df, x="X", y="Y", 
                    range_x=[-Rub,Rub], range_y=[-Rub,Rub],
                    animation_frame="S", hover_data=["I"],                    
                    color="isNew", 
                    #name = "Front",
                    # direction= "counterclockwise", start_angle=0,
                    #color_discrete_sequence=px.colors.sequential.Plasma_r, 
                    #template="plotly_dark",)
                    width=600, height=650
                    )
# fig.add_trace(go.Scatter(x=Hx, y=Hy, mode='lines', marker=dict(size=6, color=filt*1),name='Hull'))
fig.show()

In [ ]:
wat = {k:v for k,v in HSsimdata.items() if k in ['s','C']}

CvsS = pd.DataFrame(wat)
CvsS

In [ ]:
px.line(CvsS, x = 's', y = 'C')

## Casing contact

In [ ]:
", ".join(tmp.keys())

In [ ]:
I, X, Y, A, S, Nx, Ny, Ex, Ey, I1, X1, Y1, filt, Hx, Hy, ii, ti, tj, Px, Py, clas, condi, condj = tmp.values()

In [ ]:
I = np.asarray(I).ravel()[1000:]
X = np.asarray(X).ravel()[1000:]
Y = np.asarray(Y).ravel()[1000:]

In [ ]:
X.shape

In [ ]:
R,T = cart2pol(X,Y)

In [ ]:
def active(X,Y,rh):
    R,_ = cart2pol(X,Y)
    A = R < rh
    return A

In [ ]:
A = R < 2.1

In [ ]:
sum(A)

In [ ]:
px.line(R, render_mode='svg')

In [ ]:
px.scatter(x = X, y = Y, color = A , render_mode='svg', width=600, height=600)

### End

## Deriving the algorithm for rarefaction detection
The idea is to look at the segments of the curve and identify which ones are diverging the most from their neighbors. We can do this by looking at the angles between segments, or equivalently the lengths of the segments formed by non-adjacent points. The segments that are diverging the most will have the longest lengths. We can then take the top 2-5% of these segments as indicators of rarefactions, and add a point in between the endpoints of these segments to smooth out the curve.

In [ ]:
I, X,Y, Nx, Ny, Ex, Ey, *_ = tmp.values()

We'll take the results of last step from previous simulation to serve as "clay data" for the observation - idea - implementation - visualization cycle. 


In [ ]:
I = np.asarray(I).ravel()[1000:]
X = np.asarray(X).ravel()[1000:]
Y = np.asarray(Y).ravel()[1000:]
Ex = np.asarray(Ex).ravel()[1000:]
Ey = np.asarray(Ey).ravel()[1000:]
XY = np.stack((X,Y), axis=1)
XY

In [ ]:
XY.shape

Produce segments of the curve:  
- I - index of segments  
- X,Y - starting point of segments  
- c1 - end point of segments  
- v - direction vector of segments (c1 - (X,Y))  


In [ ]:
    
n = XY.shape[0]

c = XY # segment start points

c1 = np.concatenate((c[-1:,:],c[:-1]), axis=0) # segment end points 
v = c1 - c # segment direction vectors

measure lengthts of the segments 


In [ ]:
m = magn(v[:,0], v[:,1])

In [ ]:
px.line(y = m, render_mode='svg', width=600, height=600)

For every segment, calculate which quantile it's length occupies among the rest of segments.  
This gives us a measure of how "divergent" each segment is compared to the rest.  
The segments with the highest quantiles are the ones that are diverging the most from their  
neighbors, which are likely to be the rarefactions we want to identify and smooth out.


In [ ]:

# Count how many elements are strictly less than each element, then normalize
quantiles = np.sum(m[:, None] > m, axis=1) / (len(m) - 1)


In [ ]:
# Select 2% highest 

highs = quantiles > .98


In [ ]:
px.histogram(x = m, color = highs,  width=600, height=600)

In [ ]:
data = {'I': I, 'X': X, 'Y': Y, 'm': m, 'high': highs}

describe(data)

In [ ]:
df = pd.DataFrame(data)
df

Good, rarefaction points identified. Now let's visualize them and see if they make sense.

In [ ]:
px.scatter(df, x='X',y='Y', color='high', hover_data='I', render_mode='svg', width=600, height=600)

Next, we'll add points in between the endpoints of the identified segments to smooth out the curve

To add them, we use np.insert which requires the `index` where to insert, and the `values` to insert  

Small example for illustration:

In [ ]:
x = np.array([0, 1, 2, 3])
xins = np.insert(x, (0,0,2,2), (99,100, 101,102), axis=0)
print(x)
print(xins)


The `index` is the `I` of the identified points +1, since we want to insert after the identified point.

In [ ]:
newI = I[highs]
newI

The `value` is halfway from identified point along it's direction vector.  
This is a primitive interpolation scheme. More sophisticated ones should be used, but this is a good start.

In [ ]:
newXY = XY[highs] + v[highs] / 2

In [ ]:
newX, newY =  newXY[:,0], newXY[:,1]

In [ ]:
newX

In [ ]:
X1 = np.insert(X, newI[:-1], newX[:-1], axis=0)


In [ ]:
Y1 = np.insert(Y, newI[:-1], newY[:-1], axis=0)

In [ ]:
X1.shape, Y1.shape

In [ ]:
isNew = np.zeros_like(X, dtype=bool)
news = np.ones_like(newX, dtype=bool)
isNew = np.insert(isNew, newI[:-1], news[:-1], axis=0)

After insertion, points index needs to be reset.

In [ ]:
I1 = np.arange(X1.shape[0])

In [ ]:
X.shape, Y.shape, XY.shape, newX.shape, newY.shape, newI.shape

In [ ]:
data = {'I': I1, 'X': X1, 'Y': Y1, 'isNew': isNew}
describe(data)

In [ ]:
df = pd.DataFrame(data)
df

In [ ]:
px.scatter(df, x=X1,y=Y1, color=isNew, render_mode='svg', width=600, height=600)

Zooming into a corner of the shape, we can see the added points (blue) and how they smooth out the curve.
Notice however that the sharp curves appear "cut". This is due to that primitive interpolation scheme.  

The fully implemented algorithm is in `rarefactions` function in [Simulation.py](Simulation.py) file.

## Intertpolation 
### Dumb

In [ ]:
import numpy as np
import plotly.express as px 
from scipy.interpolate import Rbf, CubicSpline

In [ ]:
def interpolate(XY, xy):
    x,y = np.split(XY,2, axis=1)
    xi,_ = np.split(xy,2, axis=1)
    rbf = Rbf(x, y)
    yi = rbf(xi)
    return np.concatenate((xi,yi), axis = 1)

In [ ]:

d = 0.7
X = np.array([-1,0,1,2,3,7,8,9,12,13,14,15,20,21]) + 2
ri = np.array([4, 7, 11])
S = np.zeros_like(X)
# Y = np.sin(X)

R = 20
X0 = 3
Y = np.sqrt(R**2 - (X-X0)**2)

I = np.arange(Y.shape[0])

XY = np.stack((X,Y), axis=1)

In [ ]:
xl = np.arange(X[0], X[-1], 0.1)
yl = np.sqrt(R**2 - (xl-X0)**2)

In [ ]:
fig = px.scatter(XY ,x=0, y = 1, text = I, width=600, height=600)

fig.add_traces(
    px.line(x=xl, y = yl).data
)

fig.update_traces(textposition='top center')

In [ ]:

# direction vectors of divergents
hi = XY[ri+1,:] - XY[ri,:]

l = np.sqrt(np.sum(hi**2,axis=1))
jj = (l/d).astype(int)       

cubXY = np.zeros((0,2))
newXY = np.zeros((0,2))
newI = np.zeros(0)

for i,rr in enumerate(ri):
    j = np.arange(1, jj[i]).reshape(jj[i]-1,1) 

    wat = np.dot(j,hi[[i],:])
    
    xy = XY[rr,:] + wat / jj[i]

    newXY = np.concatenate((newXY, xy), axis=0)


    slc = slice(rr-3, rr+3)
    cXY = interpolate(XY[slc,:], xy)
    cubXY = np.concatenate((cubXY, cXY), axis=0)

    onns = np.ones(jj[i]-1)

    newI = np.concatenate((newI, onns*(rr+1)), axis=0)
    # print("a", a)
newI = newI.astype(int)
newI, newXY


In [ ]:
XY.shape, newXY.shape, newI.shape

In [ ]:
XY1 = np.insert(XY, newI, newXY,  axis=0)
XYc = np.insert(XY, newI, cubXY,  axis=0)
I1 = np.arange(XY1.shape[0])
S1 = np.insert(S, newI, np.ones_like(newI),  axis=0)

In [ ]:
fig =px.scatter(XY1, x = 0, y = 1,  color= S1.astype(str),  width=600, height=600)
fig.add_traces(
    px.line(x=xl, y = yl).data
)
fig.update_traces(textposition='top center')

In [ ]:
fig =px.scatter(XYc, x = 0, y = 1, color= S1.astype(str),  width=600, height=600)
fig.add_traces(
    px.line(x=xl, y = yl).data
)
fig.update_traces(textposition='top center')

In [ ]:
x = np.array([0, 1, 2, 3])
xins = np.insert(x, (0,0,2,2), (99,100, 101,102), axis=0)
print(x)
print(xins)

### Circular Arc

In [ ]:
def circular_arc_points(v1, v2, d):
    """
    Interpolate between two 2D vectors with points spaced along the circular arc.

    Parameters
    ----------
    v1, v2 : array_like, shape (2,)
        Vectors from the same origin and with the same length.
    d : float
        Maximum allowed arc length between adjacent returned points.

    Returns
    -------
    np.ndarray
        Array of shape (k, 2) containing the intermediate points between v1 and v2.
        If d is larger than the arc length, an empty array is returned.
    """
    v1 = np.asarray(v1, dtype=float).reshape(-1)
    v2 = np.asarray(v2, dtype=float).reshape(-1)

    if v1.shape != (2,) or v2.shape != (2,):
        raise ValueError("v1 and v2 must each be 2D vectors")
    if d <= 0:
        raise ValueError("d must be positive")

    r1 = np.linalg.norm(v1)
    r2 = np.linalg.norm(v2)
    if np.isclose(r1, 0.0) or np.isclose(r2, 0.0):
        raise ValueError("v1 and v2 must be non-zero vectors")
    if not np.isclose(r1, r2):
        raise ValueError("v1 and v2 must have the same length")

    u1 = v1 / r1
    u2 = v2 / r2

    # Signed angle from v1 to v2 (counter-clockwise if positive)
    theta = np.arctan2(u1[0] * u2[1] - u1[1] * u2[0], np.dot(u1, u2))
    if np.isclose(theta, 0.0):
        return np.empty((0, 2), dtype=float)

    arc_len = r1 * abs(theta)
    if d > arc_len:
        return np.empty((0, 2), dtype=float)

    n_segments = max(1, int(np.ceil(arc_len / d)))
    t = np.linspace(0.0, 1.0, n_segments + 1)[1:-1]
    if t.size == 0:
        return np.empty((0, 2), dtype=float)

    angles = theta * t
    c = np.cos(angles)
    s = np.sin(angles)

    points = np.column_stack((
        c * u1[0] - s * u1[1],
        s * u1[0] + c * u1[1],
    )) * r1

    return points

# Example
A = np.array([1.0, 0.0])
B = np.array([0.0, 1.0])
pts = circular_arc_points(A, B, 0.1)
pts

In [ ]:
px.scatter(pts, x=0, y=1, width=600, height=600)

In [ ]:
A = np.array([1.0, 0.0])
B = np.array([0.0, 1.0])
pts = circular_arc_points(A, B, 0.3)

In [ ]:
pts

Need a function
I have 2 2-D vectors v1, v2 from same origin and of same length. 
I want to interpolate between them with a circular arc. 
The amount of interpolation points depends on d: the maximum arc length between each adjacent pair of interpolated points.
if d > length of the arc, then 0 points is returned.
use numpy and try make it vectorized as possible.

In [ ]:
# Define vectors
A = np.array([1, 2])
B = np.array([3, 4])

# Calculate dot product
dot_product = np.dot(A, B)

# Calculate magnitudes (lengths of the vectors)
magnitude_A = np.linalg.norm(A)
magnitude_B = np.linalg.norm(B)

# Calculate angle in radians
angle_radians = np.arccos(dot_product / (magnitude_A * magnitude_B))

# Convert radians to degrees
angle_degrees = np.degrees(angle_radians)

print(f"Angle between A and B: {angle_degrees} degrees")

In [ ]:
from scipy.spatial.transform import Rotation as R
from scipy.spatial.transform import Slerp

In [ ]:
key_rots = R.random(5, random_state=2342345)
key_times = [0, 1, 2, 3, 4]

In [ ]:
key_rots.

In [ ]:
px.scatter

In [ ]:
slerp = Slerp(key_times, key_rots)

In [ ]:
times = [0, 0.5, 0.25, 1, 1.5, 2, 2.75, 3, 3.25, 3.60, 4]
interp_rots = slerp(times)

In [ ]:
key_rots.as_euler('xyz', degrees=True)